In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares

In [ ]:
# Search and download all Kepler quarters for Kepler-8
search_result = lk.search_lightcurve("Kepler-8", author="Kepler", cadence="long")
print("Search Result:", search_result)

lc_collection = search_result.download_all()
stitched_lc = lc_collection.stitch()
clean_lc = stitched_lc.remove_nans()
normalized_lc = clean_lc.normalize()

timearray = normalized_lc.time.value
fluxarray = normalized_lc.flux.value

In [ ]:
# Quick inspection of the full normalized light curve
normalized_lc.plot()
plt.show()

In [ ]:
# Set up and compute BLS periodogram
bls = BoxLeastSquares(timearray, fluxarray)
timeperiod = np.linspace(1, 10, 10000)
durations_to_try = np.linspace(0.05, 0.3, 10)

result = bls.power(timeperiod, durations_to_try)

best_index = np.argmax(result.power)
best_period = result.period[best_index]
best_duration = result.duration[best_index]
best_t0 = result.transit_time[best_index]
best_depth = result.depth[best_index]

print(f"best period: {best_period}, Best Duration: {best_duration}, Best Transit Time: {best_t0}, Best Depth: {best_depth}")
print(durations_to_try)

In [ ]:
# Periodogram plot
plt.figure()
plt.plot(result.period, result.power)
plt.xlabel("Period (days)")
plt.ylabel("BLS Power")
plt.show()

In [ ]:
# Phase folding around transit center
phase = ((timearray - best_t0 + best_period / 2) % best_period) - best_period / 2

plt.figure()
plt.scatter(phase, fluxarray, s=2, color="black", alpha=0.5)
plt.xlabel("Phase (days)")
plt.ylabel("Normalized Flux")
plt.title("Phase-Folded Light Curve")
plt.xlim(-0.5, 0.5)
plt.show()